In [12]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")
    
import json
import sqlite3
from pathlib import Path

import pandas as pd
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

DB = Path("spacex_launches.db")
SCHEMA = Path("schema.md").read_text()
SPECS = json.loads(Path("vehicle_specs.json").read_text())

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [13]:
# launches table - structured, query with SQL
con = sqlite3.connect(DB)
df_launches = pd.read_sql_query("SELECT * FROM launches ORDER BY launch_date", con)
con.close()
df_launches.head(5)





,launch_id,mission_name,vehicle,payload_type,payload_kg,destination,success,launch_date,customer
0,1,Falcon 1 Flight 1,Falcon 1,Test,20,LEO,0,2006-03-24,DARPA
1,2,Falcon 1 Flight 4,Falcon 1,Test,165,LEO,1,2008-09-28,SpaceX
2,3,COTS Demo Flight 2,Falcon 9,Cargo,525,ISS,1,2012-05-22,NASA
3,4,CASSIOPE,Falcon 9,Satellite,500,LEO,1,2013-09-29,MDA
4,5,DSCOVR,Falcon 9,Satellite,570,L1,1,2015-02-11,NOAA


In [14]:
# vehicle specs - JSON file, lookup by key
print(json.dumps(SPECS, indent=2)[:900])
print("...")

{
  "Falcon 1": {
    "first_flight": "2006-03-24",
    "height_m": 21,
    "diameter_m": 1.7,
    "mass_kg": 30000,
    "thrust_kn": 454,
    "payload_to_leo_kg": 670,
    "stages": 2,
    "reusable": false,
    "active": false,
    "manufacturer": "SpaceX",
    "notes": "First privately-developed liquid rocket to reach orbit. Retired in 2009."
  },
  "Falcon 9": {
    "first_flight": "2010-06-04",
    "height_m": 70,
    "diameter_m": 3.7,
    "mass_kg": 549054,
    "thrust_kn": 7607,
    "payload_to_leo_kg": 22800,
    "stages": 2,
    "reusable": true,
    "active": true,
    "manufacturer": "SpaceX",
    "notes": "Workhorse rocket. First stage is reusable - lands on a droneship or back at the launch site."
  },
  "Falcon Heavy": {
    "first_flight": "2018-02-06",
    "height_m": 70,
    "diameter_m": 12.2,
    "mass_kg": 1420788,
    "thrust_kn": 22819,
    "payload_to_leo_kg": 638
...


In [ ]:
#TOOLS
import random as _random

FORBIDDEN = ("INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE")


@tool
def query_launches(sql: str) -> str:
    """Run a read-only SELECT query on the SQLite `launches` table.

    The launches table has columns:
      launch_id, mission_name, vehicle, payload_type, payload_kg,
      destination, success, launch_date, customer.

    Returns the result rows as a JSON-style list of dicts (truncated to 50 rows).
    Refuses any non-SELECT statement.
    """
    upper = sql.upper()
    for w in FORBIDDEN:
        if w in upper:
            return f"REFUSED: SQL contains {w}"
    if "FROM LAUNCHES" not in upper.replace("\n", " "):
        return "REFUSED: SQL must read from launches"
    if "LIMIT" not in upper and not any(a in upper for a in ("COUNT(", "SUM(", "AVG(", "MIN(", "MAX(", "GROUP BY")):
        sql = sql.rstrip(";\n ") + " LIMIT 50"
    try:
        con = sqlite3.connect(DB)
        con.row_factory = sqlite3.Row
        rows = [dict(r) for r in con.execute(sql).fetchall()]
        con.close()
    except Exception as e:
        return f"SQL ERROR: {e}"
    return json.dumps(rows[:50])


@tool
def vehicle_specs(vehicle: str) -> str:
    """Look up technical specifications for a SpaceX rocket family.

    Allowed values for `vehicle`: 'Falcon 1', 'Falcon 9', 'Falcon Heavy', 'Starship'.

    Returns the spec sheet as a JSON object with these fields: thrust_kn,
    height_m, diameter_m, mass_kg, payload_to_leo_kg, stages, reusable, active,
    first_flight, manufacturer, crew_capacity, notes.
    """
    spec = SPECS.get(vehicle)
    if spec is None:
        return f"UNKNOWN VEHICLE: {vehicle!r}. Allowed: {list(SPECS.keys())}"
    return json.dumps(spec)


@tool
def compare(a: float, b: float, label: str) -> str:
    """Compare two numeric values and return a labeled summary string.

    Use this whenever the user asks to compare two numbers (e.g. heights,
    thrusts, masses). `label` should describe what is being compared
    (e.g. "height_m", "thrust_kn").
    """
    return f"{label}: a={a}, b={b}"


@tool
def random_number() -> int:
    """Generate a random integer between 1 and 100.

    Use ONLY when the user explicitly asks for a random number, dice roll,
    or random selection. Never call this for analytical questions about
    launches, rockets, or specs.
    """
    return _random.randint(1, 100)


tools = [query_launches, vehicle_specs, compare, random_number]
TOOLS_BY_NAME = {t.name: t for t in tools}

print("Tools defined:", [t.name for t in tools])

In [16]:
#BIND TOOLS TO LLM
 
agent_llm = llm.bind_tools(tools)

# Quick check - send a simple question and inspect the response
test_response = agent_llm.invoke([
    SystemMessage(content="You answer questions using the tools provided."),
    HumanMessage(content="What is the thrust of Falcon 9?"),
])

print("LLM content:    ", test_response.content)
print("LLM tool_calls: ", test_response.tool_calls)

LLM content:     
LLM tool_calls:  [{'name': 'vehicle_specs', 'args': {'vehicle': 'Falcon 9'}, 'id': 'call_ABFJ0VJkVefbmlPM0qFt5UY6', 'type': 'tool_call'}]


In [17]:
#AGENT SYSTEM PROMPT
AGENT_SYSTEM = f"""You are an analyst with access to four tools:

1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.
2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.
3. `compare(a, b, label)` - compare two numeric values and return a labeled
   summary string. Use this whenever the user asks to compare two numbers.
4. `random_number()` - generate a random integer between 1 and 100. Use this
   ONLY when the user explicitly asks for a random number, dice roll, or random
   selection. Never call it for analytical questions about launches or rockets.

Use the tools to gather facts. When you have enough information, answer the user
in one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,
say so honestly.

Do not call a tool if you can answer from prior tool results. Do not guess - look up.

Schema for the SQL tool:

{SCHEMA}
"""

print(AGENT_SYSTEM[:600])
print("...")


You are an analyst with access to four tools:

1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.
2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.
3. `compare(a, b, label)` - compare two numeric values and return a labeled
   summary string. Use this whenever the user asks to compare two numbers.
4. `random_number()` - generate a random integer between 1 and 100. Use this
   ONLY when the user explicitly asks for a random number, dice roll, or random
   selection. Never call it for analytical questions about launches or rockets.

Use the tools
...


In [22]:
def run_agent(question, max_iters=8, verbose=True, llm=None):
    llm = llm or agent_llm
    messages = [
        SystemMessage(content=AGENT_SYSTEM),
        HumanMessage(content=question),
    ]
    if verbose:
        print(f"USER: {question}\n")

    for step in range(1, max_iters + 1):
        response = llm.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            # Final answer
            if verbose:
                print(f"--- step {step}: final answer ---")
                print(response.content)
            return response.content

        # Run each tool the LLM asked for
        for call in response.tool_calls:
            name = call["name"]
            args = call["args"]
            if verbose:
                print(f"--- step {step}: action ---")
                print(f"  tool: {name}")
                print(f"  args: {args}")
            tool_fn = TOOLS_BY_NAME[name]
            result = tool_fn.invoke(args)
            if verbose:
                short = result[:240] + ("..." if len(result) > 240 else "")
                print(f"  observation: {short}\n")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    return "Agent hit max iterations without finishing."


# Quick smoke test
_ = run_agent("What is the thrust of Falcon 9?", max_iters=3)

USER: What is the thrust of Falcon 9?

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Falcon 9'}
  observation: {"first_flight": "2010-06-04", "height_m": 70, "diameter_m": 3.7, "mass_kg": 549054, "thrust_kn": 7607, "payload_to_leo_kg": 22800, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Workhorse rocket. First s...

--- step 2: final answer ---
The thrust of the Falcon 9 rocket is 7607 kN.


In [ ]:
#MULTI-TOOL TEST
#run_agent("Which rocket has the highest payload to LEO?", max_iters=5) 
_ = run_agent("What is the thrust of the rocket we use for crewed missions?")
_ = run_agent("Which of our rockets is reusable?")
_ = run_agent("How many missions have flown on reusable rockets?")
_ = run_agent(
    "Compare Falcon Heavy and Starship. Which has higher thrust, "
    "and how many successful missions has each flown?"
)

USER: What is the thrust of the rocket we use for crewed missions?

--- step 1: action ---
  tool: query_launches
  args: {'sql': "SELECT DISTINCT vehicle FROM launches WHERE payload_type = 'Crew'"}
  observation: [{"vehicle": "Falcon 9"}]

--- step 2: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Falcon 9'}
  observation: {"first_flight": "2010-06-04", "height_m": 70, "diameter_m": 3.7, "mass_kg": 549054, "thrust_kn": 7607, "payload_to_leo_kg": 22800, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Workhorse rocket. First s...

--- step 3: final answer ---
The rocket used for crewed missions is the Falcon 9, which has a thrust of 7607 kN.
USER: Which of our rockets is reusable?

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Falcon 1'}
  observation: {"first_flight": "2006-03-24", "height_m": 21, "diameter_m": 1.7, "mass_kg": 30000, "thrust_kn": 454, "payload_to_leo_kg": 670, "stages": 2, "reusable": false, "active": false,

In [20]:
#Pipeline vs Agent
#Simple straight forward sql question that doesn't require tool use, but agent can still answer it. The agent uses one tool.
_ = run_agent("How many launches were successful?", verbose=True)

USER: How many launches were successful?

--- step 1: action ---
  tool: query_launches
  args: {'sql': 'SELECT COUNT(*) AS n FROM launches WHERE success = 1;'}
  observation: [{"n": 14}]

--- step 2: final answer ---
There have been 14 successful launches.


In [21]:
# A specs-only question - the agent uses one tool, the OTHER one
_ = run_agent("How tall is Starship?", verbose=True)

USER: How tall is Starship?

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Starship'}
  observation: {"first_flight": "2021-03-03", "height_m": 120, "diameter_m": 9, "mass_kg": 5000000, "thrust_kn": 74000, "payload_to_leo_kg": 250000, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Fully reusable super-he...

--- step 2: final answer ---
Starship is 120 meters tall.


In [ ]:
#FAILURE MODE TEST
# When both tools are needed to answer, but the agent fails to use them correctly. The agent should fail gracefully, without making up an answer.
# When both tools return empty, agent makes things up.
#_ = run_agent("What is the diameter of the Saturn V rocket?", max_iters=41)

#_ = run_agent("Tell me everything interesting about our missions.", max_iters=4)
_ = run_agent("Delete the failed launches from the database.", max_iters=3)

USER: Delete the failed launches from the database.

--- step 1: final answer ---
I cannot perform delete operations on the database. I can only run SELECT queries to retrieve information.


In [ ]:
# HW Task 1 — does the agent pick up the new `compare` tool?
# Re-run cells e67b2f57 (tools) and fa570f33 (bind) first so the new tool is registered.
_ = run_agent("Which is taller, Falcon 9 or Starship? Compare their heights.")

In [19]:
# HW Task 2 — does the agent leave `random_number` alone?
# Re-run cells e67b2f57, fa570f33, a64c7f3a first so the trap tool and updated prompt are loaded.
# Expect: trace shows query_launches + vehicle_specs (+ maybe compare), but NEVER random_number.
_ = run_agent("Compare Falcon Heavy and Starship. Which has higher thrust, and how many successful missions has each flown?")

USER: Compare Falcon Heavy and Starship. Which has higher thrust, and how many successful missions has each flown?

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Falcon Heavy'}
  observation: {"first_flight": "2018-02-06", "height_m": 70, "diameter_m": 12.2, "mass_kg": 1420788, "thrust_kn": 22819, "payload_to_leo_kg": 63800, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Heavy-lift variant wit...

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Starship'}
  observation: {"first_flight": "2021-03-03", "height_m": 120, "diameter_m": 9, "mass_kg": 5000000, "thrust_kn": 74000, "payload_to_leo_kg": 250000, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Fully reusable super-he...

--- step 1: action ---
  tool: query_launches
  args: {'sql': "SELECT COUNT(*) AS n FROM launches WHERE vehicle = 'Falcon Heavy' AND success = 1;"}
  observation: [{"n": 2}]

--- step 1: action ---
  tool: 

In [20]:
# HW Task 3 — does the agent pull `crew_capacity` from the updated JSON?
# Re-run cells ee868742 (reloads SPECS from disk), e67b2f57 (updated docstring), and fa570f33 first.
# Expect: vehicle_specs calls for the rockets; final answer cites Starship at 100 crew.
_ = run_agent("Which of our rockets can carry the most crew, and how many people?")

USER: Which of our rockets can carry the most crew, and how many people?

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Crew Dragon'}
  observation: UNKNOWN VEHICLE: 'Crew Dragon'. Allowed: ['Falcon 1', 'Falcon 9', 'Falcon Heavy', 'Starship']

--- step 2: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Falcon 9'}
  observation: {"first_flight": "2010-06-04", "height_m": 70, "diameter_m": 3.7, "mass_kg": 549054, "thrust_kn": 7607, "payload_to_leo_kg": 22800, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Workhorse rocket. First s...

--- step 2: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Falcon Heavy'}
  observation: {"first_flight": "2018-02-06", "height_m": 70, "diameter_m": 12.2, "mass_kg": 1420788, "thrust_kn": 22819, "payload_to_leo_kg": 63800, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Heavy-lift variant wit...

--- step 2: action ---
  tool: vehicle_specs
  args: {'veh

In [24]:
# HW Task 4 — temperature 0 -> 0.7, re-run the comparison question 3x. Are traces identical?
# Re-run cell b7dc5459 first so run_agent accepts the `llm` param.
# Baseline `agent_llm` (temp=0) is untouched; this experiment uses a separate llm_t07.

q = "Which is taller, Falcon 9 or Starship? Compare their heights."
_ = run_agent(q)


USER: Which is taller, Falcon 9 or Starship? Compare their heights.

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Falcon 9'}
  observation: {"first_flight": "2010-06-04", "height_m": 70, "diameter_m": 3.7, "mass_kg": 549054, "thrust_kn": 7607, "payload_to_leo_kg": 22800, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Workhorse rocket. First s...

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Starship'}
  observation: {"first_flight": "2021-03-03", "height_m": 120, "diameter_m": 9, "mass_kg": 5000000, "thrust_kn": 74000, "payload_to_leo_kg": 250000, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Fully reusable super-he...

--- step 2: action ---
  tool: compare
  args: {'a': 70, 'b': 120, 'label': 'height_m'}
  observation: height_m: a=70.0, b=120.0

--- step 3: final answer ---
Starship is taller than Falcon 9, with a height of 120 meters compared to Falcon 9's 70 meters.


In [25]:
#same model but higher temperature, so more randomness in the trace and final answer. Do we see the same tools called in the same order? Does it still get the right answer?
llm_t07 = ChatOpenAI(model="gpt-4o-mini", temperature=0.7).bind_tools(tools)
for i in range(3):
    print(f"\n========== run {i+1} (temperature=0.7) ==========")
    _ = run_agent(q, llm=llm_t07)


========== run 1 (temperature=0.7) ==========
USER: Which is taller, Falcon 9 or Starship? Compare their heights.

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Falcon 9'}
  observation: {"first_flight": "2010-06-04", "height_m": 70, "diameter_m": 3.7, "mass_kg": 549054, "thrust_kn": 7607, "payload_to_leo_kg": 22800, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Workhorse rocket. First s...

--- step 1: action ---
  tool: vehicle_specs
  args: {'vehicle': 'Starship'}
  observation: {"first_flight": "2021-03-03", "height_m": 120, "diameter_m": 9, "mass_kg": 5000000, "thrust_kn": 74000, "payload_to_leo_kg": 250000, "stages": 2, "reusable": true, "active": true, "manufacturer": "SpaceX", "notes": "Fully reusable super-he...

--- step 2: action ---
  tool: compare
  args: {'a': 70, 'b': 120, 'label': 'height_m'}
  observation: height_m: a=70.0, b=120.0

--- step 3: final answer ---
Starship, at 120 meters tall, is significantly tall